In [6]:
import torch
from NeuralDecoder import MnistNeuralDecoder , CortexMnistDataset
import torch
from torch.utils.data import DataLoader , Dataset
from torch import nn
import torch.nn.functional as F
from ConvLSTM import ConvLSTM
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm
import copy


In [7]:
train_dataset = CortexMnistDataset('cortex_mnist/train.pt')
val_dataset   = CortexMnistDataset('cortex_mnist/val.pt')
test_dataset  = CortexMnistDataset('cortex_mnist/test.pt')
train_loader = DataLoader(train_dataset, batch_size=10, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=10, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=10, shuffle=False, num_workers=0)
device=torch.device('mps')


In [8]:
spikes_on,_,images=next(iter(test_loader))
best_model=MnistNeuralDecoder(grid_size=40)
best_model_state = torch.load('/Users/mohamed/Cortex-Image-Reconstruction/NeuralDecoder.pth')  # or wherever you saved it
best_model.load_state_dict(best_model_state)


(11, 11)


<All keys matched successfully>

In [9]:
def train(model,num_epochs:int=5):
    Epochs = num_epochs
    criterion = torch.nn.MSELoss()
    optimizer = Adam(params=model.parameters(), lr=1e-4)
    best_model = None
    best_loss = +torch.inf
    for epoch in range(Epochs):
        train_loss = 0.
        val_loss = 0.
        model.train()
        for spikes_on, _, images in tqdm(train_loader, desc=f'Epoch {epoch+1}/{Epochs}'):
            optimizer.zero_grad()
            spikes_on = spikes_on.float().to(device)
            images = images.float().to(device)
            out = model(spikes_on)
            out=out.squeeze(2)
            loss = criterion(out, images)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
        model.eval()
        with torch.no_grad():
            for spikes_on, _, images in tqdm(test_loader, desc='Validation'):
                spikes_on = spikes_on.float().to(device)
                images = images.float().to(device)

                out_val = model(spikes_on)
                loss = criterion(out_val, images)

                val_loss += loss.item()
        if val_loss < best_loss:
            print('New model saved')
            best_loss = val_loss
            best_model = copy.deepcopy(model.state_dict())

        print(f'Epoch {epoch+1}/{Epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}')

    return best_model



In [10]:
best_model.to(device)
best_model=train(best_model)

Validation: 100%|██████████| 1000/1000 [01:23<00:00, 11.99it/s]


New model saved
Epoch 1/5 | Train Loss: 158.7541 | Val Loss: 33.9724


Validation: 100%|██████████| 1000/1000 [01:25<00:00, 11.71it/s]


New model saved
Epoch 2/5 | Train Loss: 145.4493 | Val Loss: 31.0658


Validation: 100%|██████████| 1000/1000 [01:25<00:00, 11.65it/s]


New model saved
Epoch 3/5 | Train Loss: 134.7902 | Val Loss: 26.7931


Validation: 100%|██████████| 1000/1000 [01:27<00:00, 11.49it/s]


New model saved
Epoch 4/5 | Train Loss: 126.2029 | Val Loss: 25.7864


Validation: 100%|██████████| 1000/1000 [01:25<00:00, 11.74it/s]

New model saved
Epoch 5/5 | Train Loss: 119.1053 | Val Loss: 24.2350


In [11]:
torch.save(best_model,'NeuralDecoder10.pth')